In [1]:
import textwrap
from pathlib import Path
import os
import json
from outlines import Template
import requests
import langextract as lx
from rich import print as rprint
from tqdm import tqdm

In [2]:
LABEL_STUDIO_URL = 'https://cclabel.uvm.edu/'
API_KEY = os.environ['LS_TOK']
PROJ_ID = 48

headers = {'Authorization': f'Token {API_KEY}'}
export_url = f'{LABEL_STUDIO_URL}/api/projects/{PROJ_ID}/export'

params = {
    'exportType': 'JSON',
    'download_all_tasks': 'true'
}

response = requests.get(export_url, headers=headers, params=params, verify=False)

/Users/jstonge1/Documents/work/uvm/projects/llama_setup_vacc/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cclabel.uvm.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [3]:
model_config = lx.factory.ModelConfig(
      model_id="qwen3:32b", # we'll just use qwen3 - 32b, apparently it is good for NER
      provider_kwargs={
          "model_url": "http://localhost:11434",
          "format_type": lx.data.FormatType.JSON,
          "temperature": 0.1,
          "timeout": 300,  # 5 minutes
      },
  )

In [4]:
# LabelStudio client
import httpx
from label_studio_sdk.client import LabelStudio

ls = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY, httpx_client=httpx.Client(verify=False))

In [5]:
prompt = textwrap.dedent("""\
Extract faculty information including Name, Title, YearJoin, Department, and Degrees.

Provide meaningful attributes for every entity to add context and depth.
                         
For each faculty member, create separate extractions for:
    - Faculty name (extraction_class="Name")  
    - Academic title (extraction_class="Title")
    - Department (extraction_class="Department")
    - Join year (extraction_class="JoinYear") 
    - Each degree separately and where they go it (extraction_class="Degree")

Critical: Use exact text from the input for extraction_text. Do not paraphrase. Extract entities in order of appearance with no overlapping text spans. Make sure faculty names appear only once in extract_text. Ignore attributes that are missing.
                         
Note: Degrees should include the granting institution, when applicable. 
""")

examples = [
    lx.data.ExampleData(
        text=textwrap.dedent("""\
Faculty Listing
498
Department of Clinical and Diagnostic Sciences, Associate Professor,
2006, Ph.D. (Ohio State)
Childs, Gwendolyn
School of Nursing
Assistant Professor of Nursing, 2007, B.S.N. (Lander), M.S.N. (Medical
College of Georgia), Ph.D. (South Carolina)"""),
        extractions=[
            lx.data.Extraction(
                extraction_class="Name",
                extraction_text="Childs, Gwendolyn",
            ),
            lx.data.Extraction(
                extraction_class="Title",
                extraction_text="Associate Professor",
            ),
            lx.data.Extraction(
                extraction_class="Department",
                extraction_text="Clinical and Diagnostic Sciences",
            ),
            lx.data.Extraction(
                extraction_class="JoinYear",
                extraction_text="2006",
            ),
            lx.data.Extraction(
                extraction_class="Degree",
                extraction_text="B.S.N. (Lander)",
            ),
            lx.data.Extraction(
                extraction_class="Degree",
                extraction_text="M.S.N. (Medical College of Georgia)",
            ),
            lx.data.Extraction(
                extraction_class="Degree",
                extraction_text="Ph.D. (South Carolina)",
            ),
        ]
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""\
Ratchford, Brian
Lecturer in School of Business; Canisius College, B.A., 1964; University of Rochester, Ph.D., 1972."""),
        extractions=[
            lx.data.Extraction(
                extraction_class="Name",
                extraction_text="Ratchford, Brian",
            ),
            lx.data.Extraction(
                extraction_class="Title",
                extraction_text="Lecturer in School of Business",
            ),
            lx.data.Extraction(
                extraction_class="Department",
                extraction_text="School of Business",
            ),
            lx.data.Extraction(
                extraction_class="Degree",
                extraction_text="Canisius College, B.A., 1964",
            ),
            lx.data.Extraction(
                extraction_class="Degree",
                extraction_text="University of Rochester, Ph.D., 1972",
            ),
        ]
    )
]

In [6]:
def extract_predicted_labels(extracted_ents):
    predicted_label = []
    
    for i, res in enumerate(extracted_ents):
        # we requires char interval
        if res['char_interval'] is not None:
            predicted_label.append({
                "id": i,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    'start': res['char_interval']['start_pos'],
                    'end': res['char_interval']['end_pos'],
                    'score': 0.99,
                    'text': res['extraction_text'],
                    'labels': [ res['extraction_class'] ]
                }
            })
    return predicted_label

In [ ]:
DATA_DIR = Path("../data/facultyner_text")

for res in tqdm(response.json(), total=len(response.json())):
    fname = res['data']['text'].split("/")[-1]

    with open(DATA_DIR / fname)  as f:
        input_text = f.read()

    print(fname)
    print("==============")
    print(input_text)
    
    task = ls.tasks.get(res['id'])
    
    try:
        result = lx.extract(
            text_or_documents=input_text,
            prompt_description=prompt,
            examples=examples,
            config=model_config,
            use_schema_constraints=True,
            extraction_passes=1, 
            max_char_buffer=1000
        )

        # Save and reload b/c im lazy
        lx.io.save_annotated_documents([result], output_name="extraction_results.jsonl", output_dir=".")

        with open("extraction_results.jsonl") as f:
            result_json = json.loads(f.read())

        # get prediction object from langextract
        prediction = {
            "result": extract_predicted_labels(result_json['extractions'])
        }
    
    except:
        prediction = {
            "result": []
        }

    ls.annotations.create(id=task.id, **prediction)

  0%|          | 0/374 [00:00<?, ?it/s]

100663_ug_2014_2015_497_0_surya.text
Faculty Listing
498
Department of Clinical and Diagnostic Sciences, Associate Professor,
2006, Ph.D. (Ohio State)
Childs, Gwendolyn
School of Nursing
Assistant Professor of Nursing, 2007, B.S.N. (Lander), M.S.N. (Medical
College of Georgia), Ph.D. (South Carolina)
Cho, June
School of Nursing
Assistant Professor of Nursing, 2008, B.S.N. (Catholic), M.S.N. (Yonsei),
Ph.D. (North Carolina, Chapel Hill)
Cho, Won
College of Arts and
Sciences
Department of Music, Assistant Professor of Music, 2011, B.M.
(Manhattan), M.M. (Boston), D.M.A. (Memphis)
Christensen, Lois M.
School of Education
Department of Curriculum and Instruction, Professor of Early Childhood
and Elementary Education, 1996, B.A., M.A.Ed. (Arizona State), Ph.D.
(Texas AM)
Christian, Becky J.
School of Nursing
Professor of Nursing, 2009, B.S.N., M.S.N. (Missouri), Ph.D. (Texas)
Christy, Jennifer
School of Health
Braswell
Professions
Department of Rehabilitation Sciences, Assistant Professor (

/var/folders/xb/yr7ybhzx6sg2hb1smfzmt9wm0000gp/T/ipykernel_56517/2433354722.py:16: UserWarning: With 'config', schema constraints are still applied via examples. Or pass explicit schema in config.provider_kwargs.
  result = lx.extract(
2025-08-28 15:44:07,165 - langextract.debug - DEBUG - [langextract.inference] CALL: BaseLanguageModel.__init__(self=<OllamaLanguageModel>, constraint=Constraint(co...NONE: 'none'>), kwargs={})
2025-08-28 15:44:07,166 - langextract.debug - DEBUG - [langextract.inference] RETURN: BaseLanguageModel.__init__ -> None (0.0 ms)
2025-08-28 15:44:07,166 - langextract.debug - DEBUG - [langextract.inference] CALL: BaseLanguageModel.apply_schema(self=<OllamaLanguageModel>, schema_instance=<langextract....t 0x13e8c1750>)
2025-08-28 15:44:07,167 - langextract.debug - DEBUG - [langextract.inference] RETURN: BaseLanguageModel.apply_schema -> None (0.0 ms)
DEBUG:absl:Initialized Annotator with prompt:
Extract faculty information including Name, Title, YearJoin, Departmen